In [1]:
import cv2, torch, os, timm, time, warnings, gc, zipfile, telegram, sys, argparse, tqdm, matplotlib.pyplot, torchvision.transforms, pandas, numpy
from tqdm import tqdm
from torchsummary import summary
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

device = torch.device('cuda:0')
warnings.filterwarnings('ignore')

In [3]:
import pandas as pd

label_df = pd.read_csv('./seoul_dataset/train.csv')

In [4]:
import os
from glob import glob


def get_train_data(data_dir):
    img_path_list = []
    label_list = []
    img_path_list.extend(glob(os.path.join(data_dir, '*.PNG')))
    img_path_list.sort(key=lambda x: int(x.split('/')[-1].split('.')[0]))
    label_list.extend(label_df['label'])

    return img_path_list, label_list


def get_test_data(data_dir):
    img_path_list = []
    img_path_list.extend(glob(os.path.join(data_dir, '*.PNG')))
    img_path_list.sort(key=lambda x: int(x.split('/')[-1].split('.')[0]))

    return img_path_list

In [5]:
train_path, train_label = get_train_data('./seoul_dataset/train')
test_path = get_test_data('./seoul_dataset/test')

In [6]:
label_unique = sorted(numpy.unique(train_label))
label_unique = {key: value for key, value in zip(label_unique, range(len(label_unique)))}
train_labels = [label_unique[k] for k in train_label]

In [7]:
data_dir = './seoul_dataset/'

In [8]:
def img_load(path):
    img = cv2.imread(path)[:, :, ::-1]
    return img

In [9]:
train_imgs = [img_load(m) for m in tqdm(train_path)]
test_imgs = [img_load(i) for i in tqdm(test_path)]
numpy.save(data_dir + 'train_imgs', numpy.array(train_imgs))
numpy.save(data_dir + 'test_imgs', numpy.array(test_imgs))
train_imgs = numpy.load(data_dir + 'train_imgs.npy')
test_imgs = numpy.load(data_dir + 'test_imgs.npy')

100%|██████████| 199/199 [00:04<00:00, 45.85it/s]


In [10]:
def score_function(real, pred):
    score = accuracy_score(real, pred)
    return score

In [11]:
class Custom_dataset(Dataset):
    def __init__(self, img_paths, labels, mode='train'):
        self.img_paths = img_paths
        self.labels = labels
        self.mode = mode

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = self.img_paths[idx]
        if self.mode == 'train':
            train_transform = torchvision.transforms.Compose([
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Resize((224, 224))
            ])
            img = train_transform(img)
        if self.mode == 'test':
            test_transform = torchvision.transforms.Compose([
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Resize((224, 224))
            ])
            img = test_transform(img)

        label = self.labels[idx]
        return img, label

In [12]:
class Network(torch.nn.Module):
    def __init__(self, mode='train'):
        super(Network, self).__init__()
        self.mode = mode
        if self.mode == 'train':
            self.model = timm.create_model(model_name, pretrained=False, num_classes=10)
        if self.mode == 'test':
            self.model = timm.create_model(model_name, pretrained=False, num_classes=10)

    def forward(self, x):
        x = self.model(x)
        return x

In [13]:
batch_size = 64
epochs = 100
model_name = "densenet121"

In [14]:
cv = StratifiedKFold(n_splits=5, random_state=2022, shuffle=True)
pred_ensemble = []

for idx, (train_idx, val_idx) in enumerate(cv.split(train_imgs, numpy.array(train_labels))):
    print("-----------------fold_{} start!----------------".format(idx))
    t_imgs, val_imgs = train_imgs[train_idx], train_imgs[val_idx]
    t_labels, val_labels = numpy.array(train_labels)[train_idx], numpy.array(train_labels)[val_idx]

    train_dataset = Custom_dataset(numpy.array(t_imgs), numpy.array(t_labels), mode='train')
    train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
    val_dataset = Custom_dataset(numpy.array(val_imgs), numpy.array(val_labels), mode='test')
    val_loader = DataLoader(val_dataset, shuffle=True, batch_size=batch_size)

    gc.collect()
    torch.cuda.empty_cache()
    best = 0
    model = Network().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    criterion = torch.nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler()
    best_acc = 0
    early_stopping = 0

    for epoch in range(epochs):
        start = time.time()
        train_loss = 0
        train_pred = []
        train_y = []
        model.train()
        for batch in (train_loader):
            optimizer.zero_grad()
            x = torch.tensor(batch[0], dtype=torch.float32, device=device)
            y = torch.tensor(batch[1], dtype=torch.long, device=device)
            with torch.cuda.amp.autocast():
                pred = model(x)
            loss = criterion(pred, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() / len(train_loader)
            train_pred += pred.argmax(1).detach().cpu().numpy().tolist()
            train_y += y.detach().cpu().numpy().tolist()
        train_acc = score_function(train_y, train_pred)
        state_dict = model.state_dict()
        model.eval()
        with torch.no_grad():
            val_loss = 0
            val_pred = []
            val_y = []

            for batch in (val_loader):
                x_val = torch.tensor(batch[0], dtype=torch.float32, device=device)
                y_val = torch.tensor(batch[1], dtype=torch.long, device=device)
                with torch.cuda.amp.autocast():
                    pred_val = model(x_val)
                loss_val = criterion(pred_val, y_val)

                val_loss += loss_val.item() / len(val_loader)
                val_pred += pred_val.argmax(1).detach().cpu().numpy().tolist()
                val_y += y_val.detach().cpu().numpy().tolist()
            val_acc = score_function(val_y, val_pred)

            if val_acc > best_acc:
                best_epoch = epoch
                best_loss = val_loss
                best_acc = val_acc
                early_stopping = 0
                torch.save({'epoch': epoch,
                            'state_dict': state_dict,
                            'optimizer': optimizer.state_dict(),
                            'scaler': scaler.state_dict(),
                            }, data_dir + model_name + '-best_model_{}.pth'.format(idx))
                print('-----------------SAVE:{} epoch----------------'.format(best_epoch + 1))
            else:
                early_stopping += 1
            # Early Stopping
            if early_stopping == 20:
                TIME = time.time() - start
                print(
                    f'Epoch : {epoch + 1}/{epochs}, time : {TIME:.0f}s/{TIME * (epochs - epoch - 1):.0f}s, Train loss : {train_loss:.5f}, acc : {train_acc:.5f}, Val loss : {val_loss:.5f}, acc : {val_acc:.5f}')
                print("Early stopping")
                break

        TIME = time.time() - start
        print(
            f'Epoch : {epoch + 1}/{epochs}, time : {TIME:.0f}s/{TIME * (epochs - epoch - 1):.0f}s, Train loss : {train_loss:.5f}, acc : {train_acc:.5f}, Val loss : {val_loss:.5f}, acc : {val_acc:.5f}')

-----------------fold_0 start!----------------
-----------------SAVE:1 epoch----------------
Epoch : 1/100, time : 4s/423s, Train loss : 2.19492, f1 : 0.24394, Val loss : 2.39941, f1 : 0.11034
-----------------SAVE:2 epoch----------------
Epoch : 2/100, time : 4s/372s, Train loss : 1.76387, f1 : 0.69723, Val loss : 2.10938, f1 : 0.25517
-----------------SAVE:3 epoch----------------
Epoch : 3/100, time : 5s/514s, Train loss : 1.41055, f1 : 0.82007, Val loss : 1.77930, f1 : 0.67586
-----------------SAVE:4 epoch----------------
Epoch : 4/100, time : 11s/1085s, Train loss : 1.10889, f1 : 0.87543, Val loss : 1.41211, f1 : 0.82069
-----------------SAVE:5 epoch----------------
Epoch : 5/100, time : 12s/1172s, Train loss : 0.89355, f1 : 0.89792, Val loss : 0.95728, f1 : 0.85517
-----------------SAVE:6 epoch----------------
Epoch : 6/100, time : 10s/933s, Train loss : 0.66982, f1 : 0.92561, Val loss : 0.64575, f1 : 0.88276
-----------------SAVE:7 epoch----------------
Epoch : 7/100, time : 11s/

Epoch : 30/100, time : 3s/240s, Train loss : 0.01093, f1 : 1.00000, Val loss : 0.10342, f1 : 0.99310
Epoch : 31/100, time : 4s/243s, Train loss : 0.01188, f1 : 1.00000, Val loss : 0.03173, f1 : 0.99310
Epoch : 32/100, time : 3s/229s, Train loss : 0.01038, f1 : 1.00000, Val loss : 0.09802, f1 : 0.99310
Epoch : 33/100, time : 3s/226s, Train loss : 0.00894, f1 : 1.00000, Val loss : 0.04291, f1 : 0.99310
Epoch : 34/100, time : 3s/225s, Train loss : 0.00883, f1 : 1.00000, Val loss : 0.03867, f1 : 0.99310
Epoch : 35/100, time : 5s/299s, Train loss : 0.00833, f1 : 1.00000, Val loss : 0.02690, f1 : 0.99310
Epoch : 36/100, time : 3s/221s, Train loss : 0.00753, f1 : 1.00000, Val loss : 0.03642, f1 : 0.99310
Epoch : 37/100, time : 7s/471s, Train loss : 0.00750, f1 : 1.00000, Val loss : 0.02623, f1 : 0.99310
Early stopping
-----------------fold_2 start!----------------
-----------------SAVE:1 epoch----------------
Epoch : 1/100, time : 8s/789s, Train loss : 2.18398, f1 : 0.24394, Val loss : 2.4033

Epoch : 23/100, time : 10s/800s, Train loss : 0.01926, f1 : 1.00000, Val loss : 0.07816, f1 : 0.97917
Epoch : 24/100, time : 4s/297s, Train loss : 0.01790, f1 : 1.00000, Val loss : 0.14273, f1 : 0.97917
Epoch : 25/100, time : 4s/263s, Train loss : 0.01551, f1 : 1.00000, Val loss : 0.06687, f1 : 0.97917
Epoch : 26/100, time : 3s/253s, Train loss : 0.01379, f1 : 1.00000, Val loss : 0.07895, f1 : 0.97917
Epoch : 27/100, time : 3s/254s, Train loss : 0.01412, f1 : 1.00000, Val loss : 0.08064, f1 : 0.97917
Epoch : 28/100, time : 4s/252s, Train loss : 0.01345, f1 : 1.00000, Val loss : 0.07826, f1 : 0.97917
Epoch : 29/100, time : 3s/246s, Train loss : 0.01282, f1 : 1.00000, Val loss : 0.09488, f1 : 0.97917
Epoch : 30/100, time : 4s/250s, Train loss : 0.01069, f1 : 1.00000, Val loss : 0.08000, f1 : 0.97917
Epoch : 31/100, time : 3s/234s, Train loss : 0.01273, f1 : 1.00000, Val loss : 0.07643, f1 : 0.97917
Epoch : 32/100, time : 3s/235s, Train loss : 0.01015, f1 : 1.00000, Val loss : 0.07788, f1

In [15]:
pred_ensemble = []
test_dataset = Custom_dataset(numpy.array(test_imgs), numpy.array(["tmp"] * len(test_imgs)), mode='test')
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)
for i in range(5):
    model_test = Network(mode='test').to(device)
    model_test.load_state_dict(torch.load((data_dir + model_name + '-best_model_{}.pth'.format(i)))['state_dict'])
    model_test.eval()
    pred_prob = []
    with torch.no_grad():
        for batch in (test_loader):
            x = torch.tensor(batch[0], dtype=torch.float32, device=device)
            with torch.cuda.amp.autocast():
                pred = model_test(x)
                pred_prob.extend(pred.detach().cpu().numpy())
        pred_ensemble.append(pred_prob)

In [16]:
pred = (numpy.array(pred_ensemble[0]) + numpy.array(pred_ensemble[1]) + numpy.array(pred_ensemble[2]) + numpy.array(
    pred_ensemble[3]) + numpy.array(pred_ensemble[4])) / 5
f_pred = numpy.array(pred).argmax(1).tolist()
label_decoder = {val: key for key, val in label_unique.items()}
f_result = [label_decoder[result] for result in f_pred]

In [17]:
submission = pd.read_csv('./seoul_dataset/sample_submission.csv')
submission['label'] = f_result

In [18]:
submission.to_csv('./seoul_dataset/submit.csv', index=False)